# Autoencoders (vanilla / denoising)

A self-contained refresher: what an autoencoder is, the mental model, the
concepts that matter, two runnable PyTorch examples (vanilla + denoising on the
8×8 `digits` dataset), the gotchas, and how it stacks up against the
alternatives.

**Domain:** Architectures  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

An **autoencoder (AE)** is a neural network trained to copy its input to its
output *through a constriction*. It has two halves:

- **Encoder** `z = f(x)` maps the input to a low-dimensional **code** `z`.
- **Decoder** `x̂ = g(z)` maps the code back to the input space.

Training minimizes a **reconstruction loss** `L(x, x̂)` — there are no labels, so
it is **self-supervised**: the data is its own target. The trick is the
**bottleneck**: because `z` is smaller (or otherwise constrained) than `x`, the
network can't just memorize an identity map. It must discover the structure that
lets it rebuild `x` from few numbers.

**Why reach for one:**

- **Nonlinear dimensionality reduction** — a learned, nonlinear generalization of
  PCA.
- **Denoising** — train on (corrupted input → clean target) and the AE learns to
  strip noise, which forces it to learn robust, meaningful features.
- **Anomaly detection** — train on "normal" data; inputs that reconstruct badly
  (high error) are out-of-distribution.
- **Representation learning / pretraining** — use the encoder's `z` as features
  for a downstream model.

**When *not* to:** a vanilla AE is **not a generative model**. Its latent space
has no imposed structure, so sampling random `z` and decoding gives garbage. If
you need to *generate* new samples, reach for a **VAE**, **diffusion model**, or
**flow** (see the sibling notebooks). The AE is for *compressing* and *cleaning*
existing data, not inventing new data.

## 2. Mental Model

Picture an **hourglass**. Data pours in the wide top, is squeezed through a
narrow waist (the code `z`), and expands out the wide bottom:

```
   x  (64 dims)        z (16 dims)        x̂  (64 dims)
  ┌──────────┐                          ┌──────────┐
  │  input   │──encoder──▶  [ code ]  ──decoder──▶ │ output │
  └──────────┘    f             ▲          g       └──────────┘
                              bottleneck
                         (information must
                          fit through here)
```

The waist is an **information bottleneck**. To rebuild `x` from only 16 numbers,
the encoder has to throw away noise and idiosyncrasy and keep the few directions
that explain the data. That surviving summary *is* the learned representation.

For a **denoising** AE, change the picture slightly: you splash mud on the input
photo before it enters, but still grade the output against the *clean* photo. To
win, the network must learn what a clean digit actually looks like — not just
echo pixels back. Corruption is the teacher.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **Encoder / decoder** | The two halves; `f: x→z` and `g: z→x̂`. Often roughly mirror-image architectures. |
| **Latent / code / bottleneck** `z` | The compressed representation. Its dimensionality is the key knob. |
| **Reconstruction loss** | **MSE** for real-valued data; **BCE** when inputs are in `[0,1]` (pixel intensities). |
| **Undercomplete** | `dim(z) < dim(x)`. The bottleneck does the regularizing — the standard case. |
| **Overcomplete** | `dim(z) ≥ dim(x)`. Needs *extra* regularization (sparsity, denoising, noise) or it learns the trivial identity. |
| **Denoising AE (DAE)** | Input is corrupted (Gaussian noise, masking/dropout); target is the clean input. Learns robust features. |
| **Sparse / contractive AE** | Add a penalty (L1 on activations / Jacobian norm) so the code stays sparse or stable — regularizes without shrinking `dim(z)`. |
| **Tied weights** | Decoder weights are the transpose of the encoder's — fewer params, classic trick. |
| **Linear AE ≈ PCA** | A single-layer AE with linear activations and MSE loss recovers the principal-subspace of PCA. AEs earn their keep by going *nonlinear*. |

## 4. Setup

CPU is plenty for everything here — the model is tiny and `digits` has only 1,797
8×8 images.

```bash
pip install torch scikit-learn numpy matplotlib
```

In [1]:
# %pip install torch scikit-learn numpy matplotlib
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

# 8x8 handwritten digits, pixel values scaled to [0, 1].
digits = load_digits()
X = digits.data.astype("float32") / 16.0          # (1797, 64), range [0,1]
X_train, X_test = train_test_split(X, test_size=0.2, random_state=0)
X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)

print("input dim:", X.shape[1], "| train:", X_train.shape[0], "| test:", X_test.shape[0])
print("pixel range:", float(X.min()), "->", float(X.max()))

input dim: 64 | train: 1437 | test: 360
pixel range: 0.0 -> 1.0


## 5. Worked Examples

### Example 1 — A vanilla undercomplete autoencoder (and how it compares to PCA)

We squeeze 64 pixels down to a **16-dim** code and back. The decoder ends in a
`Sigmoid` because pixels live in `[0, 1]`, and we train with MSE.

In [2]:
class AutoEncoder(nn.Module):
    def __init__(self, in_dim=64, latent=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 32), nn.ReLU(),
            nn.Linear(32, latent), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 32), nn.ReLU(),
            nn.Linear(32, in_dim), nn.Sigmoid(),   # outputs in [0,1]
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def train(model, data, epochs=300, lr=1e-2, noise=0.0):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    model.train()
    for _ in range(epochs):
        inp = data + noise * torch.randn_like(data) if noise else data
        x_hat, _ = model(inp)
        loss = loss_fn(x_hat, data)        # target is always the CLEAN input
        opt.zero_grad(); loss.backward(); opt.step()
    return model


ae = train(AutoEncoder(latent=16), X_train_t, epochs=300)

ae.eval()
with torch.no_grad():
    recon, _ = ae(X_test_t)
    ae_mse = nn.functional.mse_loss(recon, X_test_t).item()
print(f"Autoencoder test reconstruction MSE: {ae_mse:.5f}")

Autoencoder test reconstruction MSE: 0.01888


Now the baseline: **PCA** with the same number of components (16). PCA gives the
*optimal* linear reconstruction for a given code size, so it's a tough baseline —
expect a small, briefly-trained AE to land in the same ballpark (often a touch
behind on pure MSE). AEs pull *ahead* when the data manifold is nonlinear and the
network is deeper; here the point is that 16 numbers already capture the digit.

In [3]:
from sklearn.decomposition import PCA

pca = PCA(n_components=16).fit(X_train)
X_test_pca = pca.inverse_transform(pca.transform(X_test))
pca_mse = float(np.mean((X_test_pca - X_test) ** 2))

print(f"PCA(16)      test reconstruction MSE: {pca_mse:.5f}")
print(f"Autoencoder  test reconstruction MSE: {ae_mse:.5f}")
print(f"variance explained by PCA(16): {pca.explained_variance_ratio_.sum():.2%}")
print("\n16 numbers reconstruct a 64-pixel digit -- a 4x compression.")

PCA(16)      test reconstruction MSE: 0.01195
Autoencoder  test reconstruction MSE: 0.01888
variance explained by PCA(16): 85.15%

16 numbers reconstruct a 64-pixel digit -- a 4x compression.


### Example 2 — A denoising autoencoder

Same architecture, but during training we **corrupt the input** with Gaussian
noise while keeping the *clean* image as the target (`noise=0.5` in our `train`
loop). At test time we feed genuinely noisy digits and see who reconstructs the
clean original better — the denoising AE or the vanilla one.

In [4]:
# Train a denoising AE: corrupt inputs, reconstruct the clean target.
dae = train(AutoEncoder(latent=16), X_train_t, epochs=300, noise=0.5)

# Build a noisy test set; the goal is to recover the CLEAN X_test.
torch.manual_seed(1)
X_test_noisy = (X_test_t + 0.5 * torch.randn_like(X_test_t)).clamp(0, 1)

dae.eval(); ae.eval()
with torch.no_grad():
    dae_out, _ = dae(X_test_noisy)
    ae_out, _ = ae(X_test_noisy)               # vanilla AE never saw noise
    dae_err = nn.functional.mse_loss(dae_out, X_test_t).item()
    ae_err = nn.functional.mse_loss(ae_out, X_test_t).item()
    raw_err = nn.functional.mse_loss(X_test_noisy, X_test_t).item()

print(f"error of the noisy input vs clean : {raw_err:.5f}  (do-nothing baseline)")
print(f"vanilla AE  denoised error        : {ae_err:.5f}")
print(f"denoising AE denoised error       : {dae_err:.5f}  <-- lowest")

error of the noisy input vs clean : 0.11878  (do-nothing baseline)
vanilla AE  denoised error        : 0.07897
denoising AE denoised error       : 0.04073  <-- lowest


The denoising AE drives the reconstruction error below both the raw noisy input
and the vanilla AE — corruption during training forced it to learn what a clean
digit *should* look like. Let's eyeball one example as ASCII art to make it
concrete.

In [5]:
def ascii_digit(vec):
    chars = " .:-=+*#%@"
    img = vec.reshape(8, 8)
    return "\n".join(
        "".join(chars[min(int(p * (len(chars) - 1)), len(chars) - 1)] for p in row)
        for row in img
    )

i = 0
with torch.no_grad():
    cleaned = dae(X_test_noisy[i:i+1])[0].squeeze().numpy()

print("CLEAN target\n" + ascii_digit(X_test[i]))
print("\nNOISY input\n" + ascii_digit(X_test_noisy[i].numpy()))
print("\nDENOISED by DAE\n" + ascii_digit(cleaned))

CLEAN target
  *@%.  
 :@**#  
 .# :%  
    **  
    @-  
   +%   
  *@@*  
  ##=#@=

NOISY input
  -.#   
 =+++@.:
  @.*@ -
=   #-  
%   @=  
 . +@ # 
 :##@%+@
 =*=@- .

DENOISED by DAE
  -#*:  
 .#***  
 .-.+*  
  .:#=  
   -#=  
  :=+=. 
  +***- 
  =##*- 


### (Optional) Example 3 — reconstruction error as an anomaly score

This cell is gated so the notebook still runs end-to-end without it. An AE
trained on one class reconstructs that class well and *everything else* badly —
that error is a ready-made out-of-distribution score.

In [6]:
import os

if os.getenv("RUN_ANOMALY_DEMO"):
    # Train only on 0s; score how badly 0s vs 8s reconstruct.
    zeros = torch.from_numpy(X_train[digits.target[:len(X_train)] == 0])
    ae0 = train(AutoEncoder(latent=8), zeros, epochs=400)
    ae0.eval()
    with torch.no_grad():
        def err(mask):
            d = torch.from_numpy(X_test[digits.target[len(X_train):][mask]])
            return nn.functional.mse_loss(ae0(d)[0], d).item()
        tgt = digits.target[len(X_train):]
        print("reconstruction error on 0s:", round(err(tgt == 0), 5))
        print("reconstruction error on 8s:", round(err(tgt == 8), 5), "(higher = anomalous)")
else:
    print("Set RUN_ANOMALY_DEMO=1 to run the anomaly-detection demo.")

Set RUN_ANOMALY_DEMO=1 to run the anomaly-detection demo.


## 6. Gotchas & Pitfalls

- **Overcomplete = identity trap.** If `dim(z) ≥ dim(x)` and there's no other
  constraint, the AE happily learns the identity function and reconstructs
  perfectly while learning *nothing*. Keep it undercomplete, or add
  regularization (denoising, sparsity, dropout).
- **Low reconstruction error ≠ useful features.** The objective rewards copying,
  not understanding. A near-perfect reconstruction can still yield a useless
  latent space. Always validate the *downstream* task (clustering, classification,
  anomaly AUC), not just the loss.
- **Match the loss to the data.** Pixels/probabilities in `[0,1]` → `Sigmoid`
  output + **BCE** (or MSE). Unbounded real values → linear output + **MSE**.
  Mismatched output activation and loss train poorly.
- **Scale your inputs, and fit the scaler on train only.** AEs are sensitive to
  feature scale. Fitting normalization on the full set leaks test info.
- **Denoising noise level is a real hyperparameter.** Too little noise ≈ vanilla
  AE; too much and the input no longer constrains the target. Tune it.
- **The latent space is not a probability distribution.** Don't sample random `z`
  and decode expecting valid samples — that's what a **VAE** adds. A vanilla AE's
  latent has holes and no notion of likelihood.
- **It's still just a copy machine without the bottleneck.** The constriction (or
  corruption, or penalty) is the whole point. Remove every constraint and you've
  trained an expensive `lambda x: x`.

## 7. When to Use vs Alternatives

| You want… | Better tool | Why |
|---|---|---|
| Fast linear dim-reduction / baseline | **PCA** | Closed-form, no training, interpretable. A linear AE just re-derives it. |
| **Nonlinear** compression / features without labels | **Autoencoder** | Captures curved manifolds PCA can't; reuse the encoder. |
| Cleaning corrupted signals (images, audio, sensor) | **Denoising AE** | Directly trained for it; learns robust features as a bonus. |
| Detecting anomalies / novelties | **Autoencoder** (reconstruction error) | Train on normal data; high error flags outliers. |
| **Generating** new samples | **VAE / diffusion / flow** | These impose latent structure / a tractable density; a vanilla AE can't sample. |
| 2-D visualization of clusters | **t-SNE / UMAP** | Purpose-built for neighborhood-preserving 2-D embeddings. |
| State-of-the-art unsupervised representations | **Self-supervised / contrastive** (SimCLR, MAE, DINO) | Usually beat plain AEs on modern transfer benchmarks (MAE is itself a masked autoencoder). |

**Rule of thumb:** AEs are the workhorse for *nonlinear compression, denoising,
and anomaly detection*. The moment you need to *generate* or need a *principled
latent distribution*, graduate to a VAE or diffusion model.

## 8. Resources

- **Deep Learning book, Ch. 14 (Autoencoders)** — Goodfellow, Bengio, Courville:
  https://www.deeplearningbook.org/contents/autoencoders.html
- **Vincent et al., "Stacked Denoising Autoencoders" (JMLR 2010)** — the
  foundational DAE paper: https://www.jmlr.org/papers/v11/vincent10a.html
- **PyTorch tutorials** (build/train models, the API used above):
  https://pytorch.org/tutorials/
- **He et al., "Masked Autoencoders Are Scalable Vision Learners" (MAE, 2021)** —
  the modern self-supervised descendant: https://arxiv.org/abs/2111.06377
- **scikit-learn PCA** (the baseline in Example 1):
  https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html

> **See also:** the `vae`, `diffusion`, and `normalizing-flows` notebooks in this
> domain for the generative cousins of the autoencoder.